In [70]:
import pandas as pd
import numpy as np
import pickle as pkl

In [71]:
df=pd.read_csv('crop_yield_dataset.csv')

In [72]:
df.columns

Index(['Temperature (°C)', 'Rainfall (mm)', 'Humidity (%)', 'Soil Type',
       'Weather Condition', 'Crop Type', 'Yield (tons/hectare)'],
      dtype='object')

In [73]:
df=df.rename(columns={"Temperature (°C)":"temperature",
                   "Rainfall (mm)":"rainfall",
                   "Humidity (%)": "humidity",
                   "Soil Type":"soil_type",
                   "Weather Condition":"weather_condition",
                   "Crop Type":"crop_type",
                   "Yield (tons/hectare)":"yield"
})

In [74]:
df.columns

Index(['temperature', 'rainfall', 'humidity', 'soil_type', 'weather_condition',
       'crop_type', 'yield'],
      dtype='object')

In [75]:
import sklearn as sk
from sklearn.model_selection import train_test_split


In [76]:
x=df.drop("yield", axis=1)  # Dropping the target variable 'yield'
y=df['yield']

x_train,x_test,y_train,y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [77]:
x_train.reset_index(drop=True, inplace=True)
x_test.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)
x_train

,temperature,rainfall,humidity,soil_type,weather_condition,crop_type
0,15.929008,828.915474,72.708730,Peaty,Sunny,Wheat
1,33.895315,121.886093,81.707230,Peaty,Rainy,Rice
2,27.212401,68.172309,65.951614,Clay,Stormy,Wheat
3,15.575654,411.028470,54.978284,Sandy,Stormy,Wheat
4,33.296918,128.095746,89.662751,Loamy,Cloudy,Corn
...,...,...,...,...,...,...
795,23.207658,396.242019,82.858963,Sandy,Rainy,Corn
796,31.187223,694.981886,45.211240,Clay,Stormy,Corn
797,30.510552,776.447446,83.338518,Clay,Cloudy,Barley
798,29.019383,29.973590,59.474095,Clay,Sunny,Barley


### One-Hot_Encoding

In [78]:
x_train_ohe=pd.get_dummies(x_train,columns=["soil_type","weather_condition","crop_type"],drop_first=True)
x_test_ohe=pd.get_dummies(x_test,columns=["soil_type","weather_condition","crop_type"],drop_first=True)

In [79]:
x_train_ohe

,temperature,rainfall,humidity,soil_type_Loamy,soil_type_Peaty,soil_type_Sandy,soil_type_Silty,weather_condition_Rainy,weather_condition_Stormy,weather_condition_Sunny,crop_type_Corn,crop_type_Rice,crop_type_Soybeans,crop_type_Wheat
0,15.929008,828.915474,72.708730,False,True,False,False,False,False,True,False,False,False,True
1,33.895315,121.886093,81.707230,False,True,False,False,True,False,False,False,True,False,False
2,27.212401,68.172309,65.951614,False,False,False,False,False,True,False,False,False,False,True
3,15.575654,411.028470,54.978284,False,False,True,False,False,True,False,False,False,False,True
4,33.296918,128.095746,89.662751,True,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,23.207658,396.242019,82.858963,False,False,True,False,True,False,False,True,False,False,False
796,31.187223,694.981886,45.211240,False,False,False,False,False,True,False,True,False,False,False
797,30.510552,776.447446,83.338518,False,False,False,False,False,False,False,False,False,False,False
798,29.019383,29.973590,59.474095,False,False,False,False,False,False,True,False,False,False,False


### Target Encoding

In [80]:
from category_encoders import LeaveOneOutEncoder

looe_encoder = LeaveOneOutEncoder(cols=["soil_type","weather_condition","crop_type"])
x_train_looe = looe_encoder.fit_transform(x_train, y_train)
x_test_looe = looe_encoder.transform(x_test)


In [81]:
x_train_looe

,temperature,rainfall,humidity,soil_type,weather_condition,crop_type
0,15.929008,828.915474,72.708730,5.947070,5.800713,6.145453
1,33.895315,121.886093,81.707230,5.977010,5.792711,5.936277
2,27.212401,68.172309,65.951614,5.646986,5.902408,6.142703
3,15.575654,411.028470,54.978284,6.020760,5.923666,6.168956
4,33.296918,128.095746,89.662751,5.980224,6.093390,5.620369
...,...,...,...,...,...,...
795,23.207658,396.242019,82.858963,6.024408,5.790415,5.650135
796,31.187223,694.981886,45.211240,5.680502,5.926932,5.650489
797,30.510552,776.447446,83.338518,5.636295,6.082929,5.869557
798,29.019383,29.973590,59.474095,5.633901,5.787198,5.867046


### pickle

In [82]:
with open('looe_encoder.pkl', 'wb') as f:
    pkl.dump(looe_encoder, f)

In [83]:
with open('looe_encoder.pkl', 'rb') as f:
    loaded_encoder = pkl.load(f)

In [84]:
loaded_encoder

,verbose,0
,cols,"['soil_type', 'weather_condition', ...]"
,drop_invariant,False
,return_df,True
,handle_unknown,'value'
,handle_missing,'value'
,random_state,None
,sigma,None


In [85]:
looe_encoder

,verbose,0
,cols,"['soil_type', 'weather_condition', ...]"
,drop_invariant,False
,return_df,True
,handle_unknown,'value'
,handle_missing,'value'
,random_state,None
,sigma,None


In [86]:
print(len(df),len(df.columns))
print(df.columns)
df_clean = df.dropna(axis=0)
df_clean = df_clean.dropna(axis=1)
df_clean =df_clean.dropna(subset=['temperature', 'rainfall'])
print(len(df_clean), len(df.columns))

1000 7
Index(['temperature', 'rainfall', 'humidity', 'soil_type', 'weather_condition',
       'crop_type', 'yield'],
      dtype='object')
1000 7


In [87]:
max(df_clean['humidity'])

89.89104278409891

### Scaling

In [88]:
numeric_columns = ['temperature', 'rainfall', 'humidity']

### Standard scaling after one-hot encoding

In [89]:
from sklearn.preprocessing import StandardScaler,minmax_scale

scaler_std= sk.preprocessing.StandardScaler()

x_train_ohe_scaled_std = scaler_std.fit_transform(x_train_ohe[['temperature', 'rainfall', 'humidity']])
x_test_ohe_scaled_std = scaler_std.transform(x_test_ohe[['temperature', 'rainfall', 'humidity']])

x_train_ohe_scaled_std_df = pd.DataFrame(x_train_ohe_scaled_std, columns=numeric_columns, index=x_train_ohe.index)
x_test_ohe_scaled_std_df = pd.DataFrame(x_test_ohe_scaled_std, columns=numeric_columns, index=x_test_ohe.index)

final_train_ohe_scaled_std = pd.concat([x_train_ohe_scaled_std_df, x_train_ohe.drop(columns=numeric_columns)], axis=1)
final_test_ohe_scaled_std = pd.concat([x_test_ohe_scaled_std_df, x_test_ohe.drop(columns=numeric_columns)], axis=1)

final_train_ohe_scaled_std

,temperature,rainfall,humidity,soil_type_Loamy,soil_type_Peaty,soil_type_Sandy,soil_type_Silty,weather_condition_Rainy,weather_condition_Stormy,weather_condition_Sunny,crop_type_Corn,crop_type_Rice,crop_type_Soybeans,crop_type_Wheat
0,-1.534059,1.077829,0.507474,False,True,False,False,False,False,True,False,False,False,True
1,1.536054,-1.345349,1.129535,False,True,False,False,True,False,False,False,True,False,False
2,0.394066,-1.529440,0.040357,False,False,False,False,False,True,False,False,False,False,True
3,-1.594441,-0.354381,-0.718223,False,False,True,False,False,True,False,False,False,False,True
4,1.433799,-1.324067,1.679497,True,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,-0.290271,-0.405058,1.209154,False,False,True,False,True,False,False,True,False,False,False
796,1.073291,0.618803,-1.393414,False,False,False,False,False,True,False,True,False,False,False
797,0.957660,0.898007,1.242305,False,False,False,False,False,False,False,False,False,False,False
798,0.702846,-1.660358,-0.407430,False,False,False,False,False,False,True,False,False,False,False


### MinMax scaling after one-hot encoding

In [90]:
from sklearn.preprocessing import MinMaxScaler
scaler_mm = MinMaxScaler()

x_train_ohe_scaled_mm = scaler_mm.fit_transform(x_train_ohe[['temperature', 'rainfall', 'humidity']])
x_test_ohe_scaled_mm = scaler_mm.transform(x_test_ohe[['temperature', 'rainfall', 'humidity']])

x_train_ohe_scaled_mm_df = pd.DataFrame(x_train_ohe_scaled_mm, columns=numeric_columns, index=x_train_ohe.index)
x_test_ohe_scaled_mm_df = pd.DataFrame(x_test_ohe_scaled_mm, columns=numeric_columns, index=x_test_ohe.index)


final_train_ohe_scaled_mm = pd.concat([x_train_ohe_scaled_mm_df, x_train_ohe.drop(columns=numeric_columns)], axis=1)
final_test_ohe_scaled_mm = pd.concat([x_test_ohe_scaled_mm_df, x_test_ohe.drop(columns=numeric_columns)], axis=1)

final_train_ohe_scaled_mm

,temperature,rainfall,humidity,soil_type_Loamy,soil_type_Peaty,soil_type_Sandy,soil_type_Silty,weather_condition_Rainy,weather_condition_Stormy,weather_condition_Sunny,crop_type_Corn,crop_type_Rice,crop_type_Soybeans,crop_type_Wheat
0,0.042025,0.828851,0.656665,False,True,False,False,False,False,True,False,False,False,True
1,0.944777,0.119121,0.837728,False,True,False,False,True,False,False,False,True,False,False
2,0.608981,0.065202,0.520702,False,False,False,False,False,True,False,False,False,False,True
3,0.024270,0.409368,0.299902,False,False,True,False,False,True,False,False,False,False,True
4,0.914709,0.125354,0.997804,True,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,0.407755,0.394525,0.860902,False,False,True,False,True,False,False,True,False,False,False
796,0.808703,0.694406,0.103375,False,False,False,False,False,True,False,True,False,False,False
797,0.774703,0.776182,0.870551,False,False,False,False,False,False,False,False,False,False,False
798,0.699776,0.026858,0.390364,False,False,False,False,False,False,True,False,False,False,False


### Standard scaling after LeaveOneOut encoding

In [91]:
scaler_std_looe = sk.preprocessing.StandardScaler()

x_train_looe_scaled_std = scaler_std_looe.fit_transform(x_train_looe[['temperature', 'rainfall', 'humidity']])
x_test_looe_scaled_std = scaler_std_looe.transform(x_test_looe[['temperature', 'rainfall', 'humidity']])

x_train_looe_scaled_std_df = pd.DataFrame(x_train_looe_scaled_std, columns=numeric_columns, index=x_train_looe.index)
x_test_looe_scaled_std_df = pd.DataFrame(x_test_looe_scaled_std, columns=numeric_columns, index=x_test_looe.index)


final_train_looe_scaled_std = pd.concat([x_train_looe_scaled_std_df, x_train_looe.drop(columns=numeric_columns)], axis=1)
final_test_looe_scaled_std = pd.concat([x_test_looe_scaled_std_df, x_test_looe.drop(columns=numeric_columns)], axis=1)

final_train_looe_scaled_std

,temperature,rainfall,humidity,soil_type,weather_condition,crop_type
0,-1.534059,1.077829,0.507474,5.947070,5.800713,6.145453
1,1.536054,-1.345349,1.129535,5.977010,5.792711,5.936277
2,0.394066,-1.529440,0.040357,5.646986,5.902408,6.142703
3,-1.594441,-0.354381,-0.718223,6.020760,5.923666,6.168956
4,1.433799,-1.324067,1.679497,5.980224,6.093390,5.620369
...,...,...,...,...,...,...
795,-0.290271,-0.405058,1.209154,6.024408,5.790415,5.650135
796,1.073291,0.618803,-1.393414,5.680502,5.926932,5.650489
797,0.957660,0.898007,1.242305,5.636295,6.082929,5.869557
798,0.702846,-1.660358,-0.407430,5.633901,5.787198,5.867046


### MinMax scaling after LeaveOneOut encoding

In [92]:
scaler_mm_looe = sk.preprocessing.MinMaxScaler()

x_train_looe_scaled_mm = scaler_mm_looe.fit_transform(x_train_looe[['temperature', 'rainfall', 'humidity']])
x_test_looe_scaled_mm = scaler_mm_looe.transform(x_test_looe[['temperature', 'rainfall', 'humidity']])

x_train_looe_scaled_mm_df = pd.DataFrame(x_train_looe_scaled_mm, columns=numeric_columns, index=x_train_looe.index)
x_test_looe_scaled_mm_df = pd.DataFrame(x_test_looe_scaled_mm, columns=numeric_columns, index=x_test_looe.index)

final_train_looe_scaled_mm = pd.concat([x_train_looe_scaled_mm_df, x_train_looe.drop(columns=numeric_columns)],axis=1)
final_test_looe_scaled_mm = pd.concat([x_test_looe_scaled_mm_df, x_test_looe.drop(columns=numeric_columns)], axis=1)

final_train_looe_scaled_mm

,temperature,rainfall,humidity,soil_type,weather_condition,crop_type
0,0.042025,0.828851,0.656665,5.947070,5.800713,6.145453
1,0.944777,0.119121,0.837728,5.977010,5.792711,5.936277
2,0.608981,0.065202,0.520702,5.646986,5.902408,6.142703
3,0.024270,0.409368,0.299902,6.020760,5.923666,6.168956
4,0.914709,0.125354,0.997804,5.980224,6.093390,5.620369
...,...,...,...,...,...,...
795,0.407755,0.394525,0.860902,6.024408,5.790415,5.650135
796,0.808703,0.694406,0.103375,5.680502,5.926932,5.650489
797,0.774703,0.776182,0.870551,5.636295,6.082929,5.869557
798,0.699776,0.026858,0.390364,5.633901,5.787198,5.867046


### pickle

In [93]:
with open('scaler_std.pkl', 'wb') as f:
    pkl.dump(scaler_std, f)

with open('scaler_mm.pkl', 'wb') as f:
    pkl.dump(scaler_mm, f)

with open('scaler_std_looe.pkl', 'wb') as f:
    pkl.dump(scaler_std_looe, f)

with open('scaler_mm_looe.pkl', 'wb') as f:
    pkl.dump(scaler_mm_looe, f)

## --

In [94]:
from sklearn.ensemble import RandomForestRegressor

In [95]:
model_se = RandomForestRegressor(n_estimators=100, random_state=42,criterion='squared_error')
model_ae = RandomForestRegressor(n_estimators=100, random_state=42,criterion='absolute_error')

model_se.fit(final_train_looe_scaled_mm, y_train)
model_ae.fit(final_train_looe_scaled_mm, y_train)

,n_estimators,100
,criterion,'absolute_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [96]:
model_se.predict(final_test_looe_scaled_mm)

array([5.73767675, 5.99244715, 5.8037662 , 5.83068781, 6.19087262,
       5.74634659, 5.57494372, 5.45849945, 5.93861051, 6.07417591,
       5.8716735 , 5.6765945 , 5.87632628, 5.94312192, 5.57922932,
       5.86369035, 5.93469732, 6.16898205, 5.97073835, 6.0825211 ,
       5.80766754, 6.04446731, 5.69090249, 5.69570282, 5.98283563,
       5.83295119, 5.88519942, 6.19697689, 6.28361052, 5.98461419,
       5.87723259, 5.862997  , 6.04436795, 5.9100188 , 5.84616416,
       5.55158832, 5.7941022 , 6.07327431, 6.48399496, 5.88890216,
       6.09771009, 5.67180926, 6.01992643, 5.6021695 , 6.08638214,
       6.23403461, 5.76001732, 5.77632832, 5.81260745, 6.20943482,
       5.73431566, 5.83615602, 6.0027069 , 5.83630156, 5.92150394,
       5.49921388, 5.95856472, 6.72712803, 5.82777192, 5.81286201,
       6.18374525, 6.08464938, 6.15121849, 6.03812008, 5.82236499,
       5.76648239, 5.81442301, 5.8756203 , 5.80201181, 6.24186984,
       6.10399646, 5.748435  , 5.99410708, 5.52859058, 5.82903

In [97]:
from sklearn.metrics import mean_squared_error

In [98]:
test_mse= mean_squared_error(y_test, model_se.predict(final_test_looe_scaled_mm))
test_mse

5.149925658031818

In [99]:
model_ae.score(final_test_looe_scaled_mm, y_test)

-0.013100919521495502

## --- Models with LOOE_MM

#### Linear Regression

In [100]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


lr_model_looe_mm = LinearRegression()
lr_model_looe_mm.fit(final_train_looe_scaled_mm, y_train)


y_pred_lr_mm = lr_model_looe_mm.predict(final_test_looe_scaled_mm)


mse_lr_model_looe_mm = mean_squared_error(y_test, y_pred_lr_mm)
r2_lr_model_looe_mm = r2_score(y_test, y_pred_lr_mm)

print(f"Linear Regression Mean Squared Error: {mse_lr_model_looe_mm}")
print(f"Linear Regression R-squared: {r2_lr_model_looe_mm}")

Linear Regression Mean Squared Error: 4.9845498932528765
Linear Regression R-squared: 0.014525817008443354


#### Support Vector Regression

In [101]:
from sklearn.svm import SVR


svr_model_looe_mm= SVR(kernel='rbf') # Radial Basis Function kernel as an example
svr_model_looe_mm.fit(final_train_looe_scaled_mm, y_train)


y_pred_svr_mm = svr_model_looe_mm.predict(final_test_looe_scaled_mm)


mse_svr_model_looe_mm = mean_squared_error(y_test, y_pred_svr_mm)
r2_svr_model_looe_mm = r2_score(y_test, y_pred_svr_mm)

print(f"Support Vector Regression Mean Squared Error: {mse_svr_model_looe_mm}")
print(f"Support Vector Regression R-squared: {r2_svr_model_looe_mm}")

Support Vector Regression Mean Squared Error: 5.0576374704729075
Support Vector Regression R-squared: 7.598262214547624e-05


#### Decision Tree Regressor

In [102]:
from sklearn.tree import DecisionTreeRegressor



dt_model_looe_mm = DecisionTreeRegressor(random_state=42)
dt_model_looe_mm.fit(final_train_looe_scaled_mm, y_train)


y_pred_dt_mm = dt_model_looe_mm.predict(final_test_looe_scaled_mm)


mse_dt_model_looe_mm = mean_squared_error(y_test, y_pred_dt_mm)
r2_dt_model_looe_mm = r2_score(y_test, y_pred_dt_mm)

print(f"Decision Tree Regressor Mean Squared Error: {mse_dt_model_looe_mm}")
print(f"Decision Tree Regressor R-squared: {r2_dt_model_looe_mm}")

Decision Tree Regressor Mean Squared Error: 5.155074660113735
Decision Tree Regressor R-squared: -0.019187910188770863


#### KNN Regressor

In [103]:
from sklearn.neighbors import KNeighborsRegressor



knn_model_looe_mm = KNeighborsRegressor(n_neighbors=5)
knn_model_looe_mm.fit(final_train_looe_scaled_mm, y_train)


y_pred_knn_mm = knn_model_looe_mm.predict(final_test_looe_scaled_mm)


mse_knn_model_looe_mm = mean_squared_error(y_test, y_pred_knn_mm)
r2_knn_model_looe_mm = r2_score(y_test, y_pred_knn_mm)

print(f"K-Nearest Neighbors Regressor Mean Squared Error: {mse_knn_model_looe_mm}")
print(f"K-Nearest Neighbors Regressor R-squared: {r2_knn_model_looe_mm}")

K-Nearest Neighbors Regressor Mean Squared Error: 5.918600975981536
K-Nearest Neighbors Regressor R-squared: -0.17014145432740557


#### Gradient Boosting

In [104]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Initialize and fit the model
gbr_model_looe_mm = GradientBoostingRegressor(n_estimators=100, random_state=42)
gbr_model_looe_mm.fit(final_train_looe_scaled_mm, y_train)

# Make predictions and evaluate
y_pred_gbr_mm = gbr_model_looe_mm.predict(final_test_looe_scaled_mm)
mse_gbr_model_looe_mm = mean_squared_error(y_test, y_pred_gbr_mm)
r2_gbr_model_looe_mm = r2_score(y_test, y_pred_gbr_mm)

print(f"Gradient Boosting Regressor Mean Squared Error: {mse_gbr_model_looe_mm}")
print(f"Gradient Boosting Regressor R-squared: {r2_gbr_model_looe_mm}")


Gradient Boosting Regressor Mean Squared Error: 5.719264016776638
Gradient Boosting Regressor R-squared: -0.13073139098781028


#### XG boost

In [105]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score


xgbr_model_looe_mm = xgb.XGBRegressor(random_state=42)
xgbr_model_looe_mm.fit(final_train_looe_scaled_mm, y_train)


y_pred_xgbr_model_looe_mm = xgbr_model_looe_mm.predict(final_test_looe_scaled_mm)
mse_xgbr_model_looe_mm = mean_squared_error(y_test, y_pred_xgbr_model_looe_mm)
r2_xgbr_model_looe_mm = r2_score(y_test, y_pred_xgbr_model_looe_mm)

print(f"XGBoost Regressor Mean Squared Error: {mse_xgbr_model_looe_mm}")
print(f"XGBoost Regressor R-squared: {r2_xgbr_model_looe_mm}")

XGBoost Regressor Mean Squared Error: 5.535892445796878
XGBoost Regressor R-squared: -0.09447777672671842


-----

## --- Models with LOOE_STD

In [106]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


lr_model_looe_std = LinearRegression()
lr_model_looe_std.fit(final_train_looe_scaled_std, y_train)


y_pred_lr_std = lr_model_looe_std.predict(final_test_looe_scaled_std)


mse_lr_model_looe_std = mean_squared_error(y_test, y_pred_lr_std)
r2_lr_model_looe_std = r2_score(y_test, y_pred_lr_std)

print(f"Linear Regression Mean Squared Error: {mse_lr_model_looe_std}")
print(f"Linear Regression R-squared: {r2_lr_model_looe_std}")

Linear Regression Mean Squared Error: 4.9845498932528765
Linear Regression R-squared: 0.014525817008443354


#### Support Vector Regression

In [107]:
from sklearn.svm import SVR

svr_model_looe_std = SVR(kernel='rbf')
svr_model_looe_std.fit(final_train_looe_scaled_std, y_train)

y_pred_svr_std = svr_model_looe_std.predict(final_test_looe_scaled_std)

mse_svr_model_looe_std = mean_squared_error(y_test, y_pred_svr_std)
r2_svr_model_looe_std = r2_score(y_test, y_pred_svr_std)

print(f"Support Vector Regression Mean Squared Error: {mse_svr_model_looe_std}")
print(f"Support Vector Regression R-squared: {r2_svr_model_looe_std}")

Support Vector Regression Mean Squared Error: 5.076285598050702
Support Vector Regression R-squared: -0.0036108594563990515


#### Decision Tree Regressor

In [108]:
from sklearn.tree import DecisionTreeRegressor

dt_model_looe_std = DecisionTreeRegressor(random_state=42)
dt_model_looe_std.fit(final_train_looe_scaled_std, y_train)

y_pred_dt_std = dt_model_looe_std.predict(final_test_looe_scaled_std)

mse_dt_model_looe_std = mean_squared_error(y_test, y_pred_dt_std)
r2_dt_model_looe_std = r2_score(y_test, y_pred_dt_std)

print(f"Decision Tree Regressor Mean Squared Error: {mse_dt_model_looe_std}")
print(f"Decision Tree Regressor R-squared: {r2_dt_model_looe_std}")

Decision Tree Regressor Mean Squared Error: 5.155074660113735
Decision Tree Regressor R-squared: -0.019187910188770863


#### KNN Regressor

In [109]:
from sklearn.neighbors import KNeighborsRegressor

knn_model_looe_std = KNeighborsRegressor(n_neighbors=5)
knn_model_looe_std.fit(final_train_looe_scaled_std, y_train)

y_pred_knn_std = knn_model_looe_std.predict(final_test_looe_scaled_std)

mse_knn_model_looe_std = mean_squared_error(y_test, y_pred_knn_std)
r2_knn_model_looe_std = r2_score(y_test, y_pred_knn_std)

print(f"K-Nearest Neighbors Regressor Mean Squared Error: {mse_knn_model_looe_std}")
print(f"K-Nearest Neighbors Regressor R-squared: {r2_knn_model_looe_std}")

K-Nearest Neighbors Regressor Mean Squared Error: 6.141473371046114
K-Nearest Neighbors Regressor R-squared: -0.21420460870267544


#### Gradient Boosting Regressor

In [110]:
from sklearn.ensemble import GradientBoostingRegressor

gbr_model_looe_std = GradientBoostingRegressor(random_state=42)
gbr_model_looe_std.fit(final_train_looe_scaled_std, y_train)

y_pred_gbr_std = gbr_model_looe_std.predict(final_test_looe_scaled_std)

mse_gbr_model_looe_std = mean_squared_error(y_test, y_pred_gbr_std)
r2_gbr_model_looe_std = r2_score(y_test, y_pred_gbr_std)

print(f"Gradient Boosting Regressor Mean Squared Error: {mse_gbr_model_looe_std}")
print(f"Gradient Boosting Regressor R-squared: {r2_gbr_model_looe_std}")

Gradient Boosting Regressor Mean Squared Error: 5.719264016776638
Gradient Boosting Regressor R-squared: -0.13073139098781028


#### XGBoost Regressor

In [111]:
import xgboost as xgb

xgb_model_looe_std = xgb.XGBRegressor(random_state=42)
xgb_model_looe_std.fit(final_train_looe_scaled_std, y_train)

y_pred_xgb_std = xgb_model_looe_std.predict(final_test_looe_scaled_std)

mse_xgb_model_looe_std = mean_squared_error(y_test, y_pred_xgb_std)
r2_xgb_model_looe_std = r2_score(y_test, y_pred_xgb_std)

print(f"XGBoost Regressor Mean Squared Error: {mse_xgb_model_looe_std}")
print(f"XGBoost Regressor R-squared: {r2_xgb_model_looe_std}")

XGBoost Regressor Mean Squared Error: 5.535892445796878
XGBoost Regressor R-squared: -0.09447777672671842


#### Lasso Regression

In [112]:
from sklearn.linear_model import Lasso

lasso_model_looe_std = Lasso(random_state=42)
lasso_model_looe_std.fit(final_train_looe_scaled_std, y_train)

y_pred_lasso_std = lasso_model_looe_std.predict(final_test_looe_scaled_std)

mse_lasso_model_looe_std = mean_squared_error(y_test, y_pred_lasso_std)
r2_lasso_model_looe_std = r2_score(y_test, y_pred_lasso_std)

print(f"Lasso Regression Mean Squared Error: {mse_lasso_model_looe_std}")
print(f"Lasso Regression R-squared: {r2_lasso_model_looe_std}")

Lasso Regression Mean Squared Error: 5.058022548461377
Lasso Regression R-squared: -1.4951098625815007e-07


#### Ridge Regression

In [113]:
from sklearn.linear_model import Ridge

ridge_model_looe_std = Ridge(random_state=42)
ridge_model_looe_std.fit(final_train_looe_scaled_std, y_train)

y_pred_ridge_std = ridge_model_looe_std.predict(final_test_looe_scaled_std)

mse_ridge_model_looe_std = mean_squared_error(y_test, y_pred_ridge_std)
r2_ridge_model_looe_std = r2_score(y_test, y_pred_ridge_std)

print(f"Ridge Regression Mean Squared Error: {mse_ridge_model_looe_std}")
print(f"Ridge Regression R-squared: {r2_ridge_model_looe_std}")

Ridge Regression Mean Squared Error: 4.98872720206899
Ridge Regression R-squared: 0.013699939029323205


#### Elastic Net Regression

In [114]:
from sklearn.linear_model import ElasticNet

elasticnet_model_looe_std = ElasticNet(random_state=42)
elasticnet_model_looe_std.fit(final_train_looe_scaled_std, y_train)

y_pred_elasticnet_std = elasticnet_model_looe_std.predict(final_test_looe_scaled_std)

mse_elasticnet_model_looe_std = mean_squared_error(y_test, y_pred_elasticnet_std)
r2_elasticnet_model_looe_std = r2_score(y_test, y_pred_elasticnet_std)

print(f"Elastic Net Regression Mean Squared Error: {mse_elasticnet_model_looe_std}")
print(f"Elastic Net Regression R-squared: {r2_elasticnet_model_looe_std}")

Elastic Net Regression Mean Squared Error: 5.058022548461377
Elastic Net Regression R-squared: -1.4951098625815007e-07


### Random Forest


In [115]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

rf_model_looe_std = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_looe_std.fit(final_train_looe_scaled_std, y_train)

y_pred_rf_std = rf_model_looe_std.predict(final_test_looe_scaled_std)

mse_rf_model_looe_std = mean_squared_error(y_test, y_pred_rf_std)
r2_rf_model_looe_std = r2_score(y_test, y_pred_rf_std)

print(f"Random Forest Regressor Mean Squared Error: {mse_rf_model_looe_std}")
print(f"Random Forest Regressor R-squared: {r2_rf_model_looe_std}")


Random Forest Regressor Mean Squared Error: 5.149925658031818
Random Forest Regressor R-squared: -0.018169922862218746


## --- Models with OHE_MM

#### Linear Regression

In [116]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

lr_model_ohe_mm = LinearRegression()
lr_model_ohe_mm.fit(final_train_ohe_scaled_mm, y_train)

y_pred_lr_model_ohe_mm = lr_model_ohe_mm.predict(final_test_ohe_scaled_mm)

mse_lr_model_ohe_mm = mean_squared_error(y_test,y_pred_lr_model_ohe_mm)
r2_lr_model_ohe_mm = r2_score(y_test,y_pred_lr_model_ohe_mm)

print(f"Linear Regression Mean Squared Error: {mse_lr_model_ohe_mm}")
print(f"Linear Regression R-squared: {r2_lr_model_ohe_mm}")

Linear Regression Mean Squared Error: 5.212936007291316
Linear Regression R-squared: -0.03062743132061896


#### Support Vector Regression

In [117]:
from sklearn.svm import SVR

svr_model_ohe_mm = SVR(kernel='rbf')
svr_model_ohe_mm.fit(final_train_ohe_scaled_mm, y_train)

y_pred_svr_mm = svr_model_ohe_mm.predict(final_test_ohe_scaled_mm)

mse_svr_model_ohe_mm = mean_squared_error(y_test, y_pred_svr_mm)
r2_svr_model_ohe_mm = r2_score(y_test, y_pred_svr_mm)

print(f"Support Vector Regression Mean Squared Error: {mse_svr_model_ohe_mm}")
print(f"Support Vector Regression R-squared: {r2_svr_model_ohe_mm}")

Support Vector Regression Mean Squared Error: 5.657591047621245
Support Vector Regression R-squared: -0.11853829026805562


#### Decision Tree Regressor

In [118]:
from sklearn.tree import DecisionTreeRegressor

dt_model_ohe_mm = DecisionTreeRegressor(random_state=42)
dt_model_ohe_mm.fit(final_train_ohe_scaled_mm, y_train)

y_pred_dt_mm = dt_model_ohe_mm.predict(final_test_ohe_scaled_mm)

mse_dt_model_ohe_mm = mean_squared_error(y_test, y_pred_dt_mm)
r2_dt_model_ohe_mm = r2_score(y_test, y_pred_dt_mm)

print(f"Decision Tree Regressor Mean Squared Error: {mse_dt_model_ohe_mm}")
print(f"Decision Tree Regressor R-squared: {r2_dt_model_ohe_mm}")

Decision Tree Regressor Mean Squared Error: 8.484407223077232
Decision Tree Regressor R-squared: -0.6774161068479687


#### KNN Regressor

In [119]:
from sklearn.neighbors import KNeighborsRegressor

knn_model_ohe_mm = KNeighborsRegressor(n_neighbors=5)
knn_model_ohe_mm.fit(final_train_ohe_scaled_mm, y_train)

y_pred_knn_mm = knn_model_ohe_mm.predict(final_test_ohe_scaled_mm)

mse_knn_model_ohe_mm = mean_squared_error(y_test, y_pred_knn_mm)
r2_knn_model_ohe_mm = r2_score(y_test, y_pred_knn_mm)

print(f"K-Nearest Neighbors Regressor Mean Squared Error: {mse_knn_model_ohe_mm}")
print(f"K-Nearest Neighbors Regressor R-squared: {r2_knn_model_ohe_mm}")

K-Nearest Neighbors Regressor Mean Squared Error: 6.517204629028344
K-Nearest Neighbors Regressor R-squared: -0.2884888394585221


#### Gradient Boosting Regressor

In [120]:
from sklearn.ensemble import GradientBoostingRegressor

gbr_model_ohe_mm = GradientBoostingRegressor(random_state=42)
gbr_model_ohe_mm.fit(final_train_ohe_scaled_mm, y_train)

y_pred_gbr_mm = gbr_model_ohe_mm.predict(final_test_ohe_scaled_mm)

mse_gbr_model_ohe_mm = mean_squared_error(y_test, y_pred_gbr_mm)
r2_gbr_model_ohe_mm = r2_score(y_test, y_pred_gbr_mm)

print(f"Gradient Boosting Regressor Mean Squared Error: {mse_gbr_model_ohe_mm}")
print(f"Gradient Boosting Regressor R-squared: {r2_gbr_model_ohe_mm}")

Gradient Boosting Regressor Mean Squared Error: 5.491802116720347
Gradient Boosting Regressor R-squared: -0.08576086507872027


#### XGBoost Regressor

In [121]:
import xgboost as xgb

xgb_model_ohe_mm = xgb.XGBRegressor(random_state=42)
xgb_model_ohe_mm.fit(final_train_ohe_scaled_mm, y_train)

y_pred_xgb_mm = xgb_model_ohe_mm.predict(final_test_ohe_scaled_mm)

mse_xgb_model_ohe_mm = mean_squared_error(y_test, y_pred_xgb_mm)
r2_xgb_model_ohe_mm = r2_score(y_test, y_pred_xgb_mm)

print(f"XGBoost Regressor Mean Squared Error: {mse_xgb_model_ohe_mm}")
print(f"XGBoost Regressor R-squared: {r2_xgb_model_ohe_mm}")

XGBoost Regressor Mean Squared Error: 6.257526985873885
XGBoost Regressor R-squared: -0.23714907584712575


#### Lasso Regression

In [122]:
from sklearn.linear_model import Lasso

lasso_model_ohe_mm = Lasso(random_state=42)
lasso_model_ohe_mm.fit(final_train_ohe_scaled_mm, y_train)

y_pred_lasso_mm = lasso_model_ohe_mm.predict(final_test_ohe_scaled_mm)

mse_lasso_model_ohe_mm = mean_squared_error(y_test, y_pred_lasso_mm)
r2_lasso_model_ohe_mm = r2_score(y_test, y_pred_lasso_mm)

print(f"Lasso Regression Mean Squared Error: {mse_lasso_model_ohe_mm}")
print(f"Lasso Regression R-squared: {r2_lasso_model_ohe_mm}")

Lasso Regression Mean Squared Error: 5.058022548461377
Lasso Regression R-squared: -1.4951098625815007e-07


#### Ridge Regression

In [123]:
from sklearn.linear_model import Ridge

ridge_model_ohe_mm = Ridge(random_state=42)
ridge_model_ohe_mm.fit(final_train_ohe_scaled_mm, y_train)

y_pred_ridge_mm = ridge_model_ohe_mm.predict(final_test_ohe_scaled_mm)

mse_ridge_model_ohe_mm = mean_squared_error(y_test, y_pred_ridge_mm)
r2_ridge_model_ohe_mm = r2_score(y_test, y_pred_ridge_mm)

print(f"Ridge Regression Mean Squared Error: {mse_ridge_model_ohe_mm}")
print(f"Ridge Regression R-squared: {r2_ridge_model_ohe_mm}")

Ridge Regression Mean Squared Error: 5.208164565647947
Ridge Regression R-squared: -0.029684089864341123


#### Elastic Net Regression

In [124]:
from sklearn.linear_model import ElasticNet

elasticnet_model_ohe_mm = ElasticNet(random_state=42)
elasticnet_model_ohe_mm.fit(final_train_ohe_scaled_mm, y_train)

y_pred_elasticnet_mm = elasticnet_model_ohe_mm.predict(final_test_ohe_scaled_mm)

mse_elasticnet_model_ohe_mm = mean_squared_error(y_test, y_pred_elasticnet_mm)
r2_elasticnet_model_ohe_mm = r2_score(y_test, y_pred_elasticnet_mm)

print(f"Elastic Net Regression Mean Squared Error: {mse_elasticnet_model_ohe_mm}")
print(f"Elastic Net Regression R-squared: {r2_elasticnet_model_ohe_mm}")

Elastic Net Regression Mean Squared Error: 5.058022548461377
Elastic Net Regression R-squared: -1.4951098625815007e-07


#### Random Forest Regressor

In [125]:
from sklearn.ensemble import RandomForestRegressor

rf_model_ohe_mm = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_ohe_mm.fit(final_train_ohe_scaled_mm, y_train)

y_pred_rf_mm = rf_model_ohe_mm.predict(final_test_ohe_scaled_mm)

mse_rf_model_ohe_mm = mean_squared_error(y_test, y_pred_rf_mm)
r2_rf_model_ohe_mm = r2_score(y_test, y_pred_rf_mm)

print(f"Random Forest Regressor Mean Squared Error: {mse_rf_model_ohe_mm}")
print(f"Random Forest Regressor R-squared: {r2_rf_model_ohe_mm}")

Random Forest Regressor Mean Squared Error: 5.191082047437552
Random Forest Regressor R-squared: -0.02630677776247725


## --- Models with OHE_STD

#### Linear Regression

In [126]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

lr_model_ohe_std = LinearRegression()
lr_model_ohe_std.fit(final_train_ohe_scaled_std, y_train)

y_pred_lr_model_ohe_std = lr_model_ohe_std.predict(final_test_ohe_scaled_std)

mse_lr_model_ohe_std = mean_squared_error(y_test, y_pred_lr_model_ohe_std)
r2_lr_model_ohe_std = r2_score(y_test, y_pred_lr_model_ohe_std)

print(f"Linear Regression Mean Squared Error: {mse_lr_model_ohe_std}")
print(f"Linear Regression R-squared: {r2_lr_model_ohe_std}")

Linear Regression Mean Squared Error: 5.212936007291315
Linear Regression R-squared: -0.030627431320618737


#### Support Vector Regression

In [127]:
from sklearn.svm import SVR

svr_model_ohe_std = SVR(kernel='rbf')
svr_model_ohe_std.fit(final_train_ohe_scaled_std, y_train)

y_pred_svr_std = svr_model_ohe_std.predict(final_test_ohe_scaled_std)

mse_svr_model_ohe_std = mean_squared_error(y_test, y_pred_svr_std)
r2_svr_model_ohe_std = r2_score(y_test, y_pred_svr_std)

print(f"Support Vector Regression Mean Squared Error: {mse_svr_model_ohe_std}")
print(f"Support Vector Regression R-squared: {r2_svr_model_ohe_std}")

Support Vector Regression Mean Squared Error: 5.442355266056477
Support Vector Regression R-squared: -0.07598493830438069


#### Decision Tree Regressor

In [128]:
from sklearn.tree import DecisionTreeRegressor

dt_model_ohe_std = DecisionTreeRegressor(random_state=42)
dt_model_ohe_std.fit(final_train_ohe_scaled_std, y_train)

y_pred_dt_std = dt_model_ohe_std.predict(final_test_ohe_scaled_std)

mse_dt_model_ohe_std = mean_squared_error(y_test, y_pred_dt_std)
r2_dt_model_ohe_std = r2_score(y_test, y_pred_dt_std)

print(f"Decision Tree Regressor Mean Squared Error: {mse_dt_model_ohe_std}")
print(f"Decision Tree Regressor R-squared: {r2_dt_model_ohe_std}")

Decision Tree Regressor Mean Squared Error: 8.484407223077232
Decision Tree Regressor R-squared: -0.6774161068479687


#### KNN Regressor

In [129]:
from sklearn.neighbors import KNeighborsRegressor

knn_model_ohe_std = KNeighborsRegressor(n_neighbors=5)
knn_model_ohe_std.fit(final_train_ohe_scaled_std, y_train)

y_pred_knn_std = knn_model_ohe_std.predict(final_test_ohe_scaled_std)

mse_knn_model_ohe_std = mean_squared_error(y_test, y_pred_knn_std)
r2_knn_model_ohe_std = r2_score(y_test, y_pred_knn_std)

print(f"K-Nearest Neighbors Regressor Mean Squared Error: {mse_knn_model_ohe_std}")
print(f"K-Nearest Neighbors Regressor R-squared: {r2_knn_model_ohe_std}")

K-Nearest Neighbors Regressor Mean Squared Error: 5.751750870730666
K-Nearest Neighbors Regressor R-squared: -0.1371542288656391


#### Gradient Boosting Regressor

In [130]:
from sklearn.ensemble import GradientBoostingRegressor

gbr_model_ohe_std = GradientBoostingRegressor(random_state=42)
gbr_model_ohe_std.fit(final_train_ohe_scaled_std, y_train)

y_pred_gbr_std = gbr_model_ohe_std.predict(final_test_ohe_scaled_std)

mse_gbr_model_ohe_std = mean_squared_error(y_test, y_pred_gbr_std)
r2_gbr_model_ohe_std = r2_score(y_test, y_pred_gbr_std)

print(f"Gradient Boosting Regressor Mean Squared Error: {mse_gbr_model_ohe_std}")
print(f"Gradient Boosting Regressor R-squared: {r2_gbr_model_ohe_std}")

Gradient Boosting Regressor Mean Squared Error: 5.491802116720347
Gradient Boosting Regressor R-squared: -0.08576086507872027


#### XGBoost Regressor

In [131]:
import xgboost as xgb

xgb_model_ohe_std = xgb.XGBRegressor(random_state=42)
xgb_model_ohe_std.fit(final_train_ohe_scaled_std, y_train)

y_pred_xgb_std = xgb_model_ohe_std.predict(final_test_ohe_scaled_std)

mse_xgb_model_ohe_std = mean_squared_error(y_test, y_pred_xgb_std)
r2_xgb_model_ohe_std = r2_score(y_test, y_pred_xgb_std)

print(f"XGBoost Regressor Mean Squared Error: {mse_xgb_model_ohe_std}")
print(f"XGBoost Regressor R-squared: {r2_xgb_model_ohe_std}")

XGBoost Regressor Mean Squared Error: 6.257526985873885
XGBoost Regressor R-squared: -0.23714907584712575


#### Lasso Regression

In [132]:
from sklearn.linear_model import Lasso

lasso_model_ohe_std = Lasso(random_state=42)
lasso_model_ohe_std.fit(final_train_ohe_scaled_std, y_train)

y_pred_lasso_std = lasso_model_ohe_std.predict(final_test_ohe_scaled_std)

mse_lasso_model_ohe_std = mean_squared_error(y_test, y_pred_lasso_std)
r2_lasso_model_ohe_std = r2_score(y_test, y_pred_lasso_std)

print(f"Lasso Regression Mean Squared Error: {mse_lasso_model_ohe_std}")
print(f"Lasso Regression R-squared: {r2_lasso_model_ohe_std}")

Lasso Regression Mean Squared Error: 5.058022548461377
Lasso Regression R-squared: -1.4951098625815007e-07


#### Ridge Regression

In [133]:
from sklearn.linear_model import Ridge

ridge_model_ohe_std = Ridge(random_state=42)
ridge_model_ohe_std.fit(final_train_ohe_scaled_std, y_train)

y_pred_ridge_std = ridge_model_ohe_std.predict(final_test_ohe_scaled_std)

mse_ridge_model_ohe_std = mean_squared_error(y_test, y_pred_ridge_std)
r2_ridge_model_ohe_std = r2_score(y_test, y_pred_ridge_std)

print(f"Ridge Regression Mean Squared Error: {mse_ridge_model_ohe_std}")
print(f"Ridge Regression R-squared: {r2_ridge_model_ohe_std}")

Ridge Regression Mean Squared Error: 5.208524726608942
Ridge Regression R-squared: -0.029755295757828426


#### Random Forest Regressor

In [134]:
from sklearn.ensemble import RandomForestRegressor

rf_model_ohe_std = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_ohe_std.fit(final_train_ohe_scaled_std, y_train)

y_pred_rf_std = rf_model_ohe_std.predict(final_test_ohe_scaled_std)

mse_rf_model_ohe_std = mean_squared_error(y_test, y_pred_rf_std)
r2_rf_model_ohe_std = r2_score(y_test, y_pred_rf_std)

print(f"Random Forest Regressor Mean Squared Error: {mse_rf_model_ohe_std}")
print(f"Random Forest Regressor R-squared: {r2_rf_model_ohe_std}")

Random Forest Regressor Mean Squared Error: 5.191082047437552
Random Forest Regressor R-squared: -0.02630677776247725


## Pipeline 

In [135]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# Define column groups
categorical_cols = ["soil_type", "weather_condition", "crop_type"]


# Define transformers
categorical_transformer = LeaveOneOutEncoder(cols=categorical_cols)
numerical_transformer = MinMaxScaler()

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('encode', categorical_transformer, categorical_cols),
        ('scale', numerical_transformer, numeric_columns)
    ]
)

# Create pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Fit and evaluate
pipeline.fit(x_train, y_train)


,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('encode', ...), ('scale', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [136]:
new_data = pd.DataFrame([{
    "soil_type": "Clay",
    "weather_condition": "Sunny",
    "crop_type": "Soybeans",
    "temperature": 18.4104,
    "rainfall": 596.2698,
    "humidity": 68.266
}])

y_new_pred = pipeline.predict(new_data)
print("Predicted Yield:", y_new_pred[0])


Predicted Yield: 6.364934555201266


#### pickle pipeline

In [137]:
with open('pipeline.pkl', 'wb') as f:
    pkl.dump(pipeline, f)